In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
def generate_polymerChain(N_bonds, bond_length_sqd, dim):
    # mean 0 because of space isotropicity    
    bond_vectors = np.random.normal(0, np.sqrt(bond_length_sqd/dim), size=(N_bonds, dim))
    
    # Generate polymer chain from bond vectors
    polymerChain = []
    
    # Could replace by np.cumsum
    # but the code below works, so why bother! :p
    for idx, bond in enumerate(bond_vectors):
        if idx == 0:
            polymerChain.append(bond)
        else:
            polymerChain.append(polymerChain[idx-1] + bond)
            
    return polymerChain

# count the number of occurences of target in data for some tolerance
def count_close(data, target, tolerance):
    # Distances between the target vector and the data
    distances = np.linalg.norm(data - target, axis=1)

    return np.sum(distances <= tolerance)

# Computes the analytical free energy for some R defined accordingly
def compute_F_analytical(N_bonds, R, bond_length_std, dim):
    return (dim*R**2)/(2*N_bonds*bond_length_sqd)

In [ ]:
# Space dimension
dim = 3

# Number of segments/bonds
N_bonds = 50

# Bond parameters
bond_length_sqd = 1

# Ensemble size
N_ensemble = 1000000

In [ ]:
# Stores end-to-end distance vector for different realizations
end_to_end_vector = []
for ensemble in tqdm(range(N_ensemble)):
    chain = generate_polymerChain(N_bonds, bond_length_sqd, dim) # Generate chain
    end_to_end_vector.append(chain[-1]-chain[0])

In [ ]:
# Plot the histogram of end-to-end vector distribution for every spatial component in different realisations of the polymer chain
fig, ax = plt.subplots(figsize = (4, 3))
ax.hist([e[0] for e in end_to_end_vector], histtype = "step", label = "x", density=True)
ax.hist([e[1] for e in end_to_end_vector], histtype = "step", label = "y", density=True)
ax.hist([e[2] for e in end_to_end_vector], histtype = "step", label = "z", density=True);
ax.legend()
ax.set_xlabel(r"$\mathbf{r}_{N-1} - \mathbf{r}_{0}$")
ax.set_ylabel("density");

In [ ]:
# Plot the distance distribution in different realisations of the polymer chain
end_to_end_distances = []
for _ in end_to_end_vector:
    end_to_end_distances.append(np.linalg.norm(_))

bin_width = 0.1
bins = np.arange(np.min(end_to_end_distances), np.max(end_to_end_distances) + bin_width, bin_width)
hist, bin_edges = np.histogram(end_to_end_distances, bins, density=True)
bin_centers = (bin_edges[:1] + bin_edges[1:])/2

fig, ax = plt.subplots(figsize = (4, 3))
ax.plot(bin_centers, hist)
ax.set_xlabel(r"$\left|\mathbf{r}_{N-1} - \mathbf{r}_{0}\right|$")
ax.set_ylabel("density");
ax.set_ylim([0,np.max(hist)]);

In [ ]:
# Initial state
R_in = np.array([0, 0, 0])

# Final state(s)
R_fis = []

for R_fi in np.arange(0, np.max(bin_centers), 1):
    R_fis.append(np.array([R_fi, 0, 0]))
    

In [ ]:
# Calculate the free energy difference between the initial and the final states
Fs = []

tolerance = 0.5
# Number of occurences of initial position
counts_in = count_close(end_to_end_vector, R_in, tolerance)

# Number of occurences of final position
for R_fi in R_fis:
    counts_fi = count_close(end_to_end_vector, R_fi, tolerance)
    F = -np.log(counts_fi/counts_in)
    Fs.append(F)

In [ ]:
F_analytical = []
x = np.arange(np.min(end_to_end_distances), np.max(end_to_end_distances)/2, tolerance)
for R in x:
    F_analytical.append(compute_F_analytical(N_bonds, R, bond_length_sqd, dim))

In [ ]:
fig, ax = plt.subplots(figsize = (4, 3))
ax.plot(x, F_analytical, label = "analytical")
ax.scatter([R_fi[0] for R_fi in R_fis], Fs, label = f"simulation (tol = {tolerance})", color = "k", marker = "x");
ax.set_xlabel(r"$R$")
ax.set_ylabel(r"$\Delta F$")
ax.legend()
ax.set_xticks([R_fi[0] for R_fi in R_fis]);